# HyperView — Colab Smoke Test

This notebook verifies that HyperView can launch inside Google Colab and open the UI in a new browser tab.

The smoke test downloads a small set of public-domain NASA images and computes real embeddings using a lightweight model.

In [ ]:
%pip install hyperview

In [ ]:
import hyperview as hv

In [ ]:
import json
import random
import urllib.parse
import urllib.request
from pathlib import Path

# Download a small set of NASA space images (public domain)
NUM_IMAGES = 24
CACHE_DIR = Path("/tmp/_nasa_smoke_images")
NASA_QUERIES = ["black hole", "nebula", "galaxy", "jwst", "hubble"]

CACHE_DIR.mkdir(parents=True, exist_ok=True)
image_files = [
    p for p in CACHE_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
]
if len(image_files) < NUM_IMAGES:
    rng = random.Random(42)
    query = rng.choice(NASA_QUERIES)
    params = {"q": query, "media_type": "image", "page": "1"}
    api_url = f"https://images-api.nasa.gov/search?{urllib.parse.urlencode(params)}"
    items = json.load(urllib.request.urlopen(api_url))["collection"]["items"]
    rng.shuffle(items)

    for i, item in enumerate(items[:NUM_IMAGES]):
        href = item["links"][0]["href"]
        ext = Path(urllib.parse.urlparse(href).path).suffix or ".jpg"
        urllib.request.urlretrieve(href, CACHE_DIR / f"nasa_{i:03d}{ext}")

# Create dataset and add images
dataset = hv.Dataset("colab_smoke", persist=False)
added, skipped = dataset.add_images_dir(str(CACHE_DIR), label_from_folder=False)
print(f"✓ Loaded {added} NASA images" + (f" ({skipped} already present)" if skipped else ""))

# Compute embeddings with a lightweight model
MODEL = "google/siglip-base-patch16-224"
dataset.compute_embeddings(model=MODEL, show_progress=True)
dataset.compute_visualization()

# Launch HyperView
hv.launch(dataset, port=6262)

Click the link printed above to open HyperView in a new tab. In Colab, you may need to copy/paste the URL.